# Лабораторная 05. repartition, coalesce и output files

Цель: увидеть, как partitions влияют на shuffle и количество parquet-файлов.

In [1]:
from pathlib import Path
import shutil
from pyspark.sql import SparkSession

spark = (SparkSession.builder.appName('lab-05-files').master('local[*]')
    .config('spark.driver.memory', '2g')
    .config('spark.sql.shuffle.partitions', '8')
    .config('spark.sql.adaptive.enabled', 'false')
    .getOrCreate())
spark.sparkContext.setLogLevel('WARN')
base = Path('spark_core_data').absolute()
orders = spark.read.parquet((base / 'orders').as_uri())
out_base = base / 'lab05_output'
if out_base.exists():
    shutil.rmtree(out_base)
out_base.mkdir(parents=True, exist_ok=True)
print('Spark UI:', spark.sparkContext.uiWebUrl)

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/07/07 10:19:37 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
                                                                                

Spark UI: http://0a370e2ebe67:4040


In [3]:
def count_parquet_files(path):
    return len(list(Path(path).glob('*.parquet')))

def write_and_count(df, name):
    path = out_base / name
    df.write.mode('overwrite').parquet(path.as_uri())
    return count_parquet_files(path)


## Базовая запись

In [4]:
orders.rdd.getNumPartitions(), write_and_count(orders, 'base')

(4, 4)

## `repartition(20)`
Проверьте explain: должен появиться Exchange.

In [5]:
orders_20 = orders.repartition(20)
orders_20.explain('formatted')
orders_20.rdd.getNumPartitions(), write_and_count(orders_20, 'repartition_20')

== Physical Plan ==
Exchange (3)
+- * ColumnarToRow (2)
   +- Scan parquet  (1)


(1) Scan parquet 
Output [5]: [order_id#0L, customer_id#1L, order_date#2, status#3, order_amount#4]
Batched: true
Location: InMemoryFileIndex [file:/materials/seminar_04_spark_core/practice/spark_core_data/orders]
ReadSchema: struct<order_id:bigint,customer_id:bigint,order_date:date,status:string,order_amount:decimal(10,2)>

(2) ColumnarToRow [codegen id : 1]
Input [5]: [order_id#0L, customer_id#1L, order_date#2, status#3, order_amount#4]

(3) Exchange
Input [5]: [order_id#0L, customer_id#1L, order_date#2, status#3, order_amount#4]
Arguments: RoundRobinPartitioning(20), REPARTITION_BY_NUM, [plan_id=39]




(20, 20)

## `coalesce(2)`
Обычно дешевле, но может дать менее равномерные partitions.

In [6]:
orders_2 = orders.coalesce(2)
orders_2.explain('formatted')
orders_2.rdd.getNumPartitions(), write_and_count(orders_2, 'coalesce_2')

== Physical Plan ==
Coalesce (3)
+- * ColumnarToRow (2)
   +- Scan parquet  (1)


(1) Scan parquet 
Output [5]: [order_id#0L, customer_id#1L, order_date#2, status#3, order_amount#4]
Batched: true
Location: InMemoryFileIndex [file:/materials/seminar_04_spark_core/practice/spark_core_data/orders]
ReadSchema: struct<order_id:bigint,customer_id:bigint,order_date:date,status:string,order_amount:decimal(10,2)>

(2) ColumnarToRow [codegen id : 1]
Input [5]: [order_id#0L, customer_id#1L, order_date#2, status#3, order_amount#4]

(3) Coalesce
Input [5]: [order_id#0L, customer_id#1L, order_date#2, status#3, order_amount#4]
Arguments: 2




(2, 2)

Вопросы:

- Сколько файлов получилось в каждом каталоге? в каталоге orders_20 - 20, orders_2 - 2
- Почему количество файлов связано с partitions? потому что каждая партиция записывается в один файл
- Где был shuffle? только в repartition, в coalesce не было 
- Почему `repartition` дороже `coalesce`? из-за exchange - это дорогая операция, которая требует перемещение данных между исполнителями
- Почему `coalesce(1)` может вызвать проблемы? потому что все данные будут записаны в одной партиции и значит пропадает весь смысл распределенной системы, исчезает параллелизм 

In [7]:
spark.stop()